In [1]:
# --- Tag every coded opportunity with typical education needed for entry ---
#
# Source: BLS Employment Projections Table 5.4, "Education and training
# assignments by detailed occupation" - keyed by SOC code, which matches
# our existing "SOC Code" column format exactly (e.g. "15-1255").
# https://www.bls.gov/emp/tables/education-and-training-by-occupation.htm
#
# I can't reach bls.gov from my own environment to test this against the
# live file, so this is built defensively against the confirmed table
# structure (found via web search/fetch) rather than verified end-to-end -
# run this for real and check the printed summary before trusting it.

import pandas as pd
import requests
from io import BytesIO

EDUCATION_XLSX_URL = "https://www.bls.gov/emp/ind-occ-matrix/education.xlsx"
CODED_PATH = "../data/AllRoles_nioccs_coded.csv"
OUT_PATH = "../data/AllRoles_nioccs_coded_with_education.csv"

# If you'd rather just download education.xlsx yourself (browser, not
# Python) and drop it in your data folder, set this to that local path
# and the script will read it directly instead of hitting the URL at all.
# BLS's site blocks plain scripted requests (403) fairly often regardless
# of headers, so this is the more reliable option if the download below
# still fails.
LOCAL_XLSX_PATH = "../data/education.xlsx"  # e.g. "../../data/education.xlsx"

# A real browser User-Agent - bls.gov's bot protection blocks requests'
# default UA string outright, which is almost certainly the 403 you saw.
REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

# Ordered from highest to lowest - used to build the boolean cut points below.
EDUCATION_LEVELS_DESCENDING = [
    "Doctoral or professional degree",
    "Master's degree",
    "Bachelor's degree",
    "Associate's degree",
    "Postsecondary nondegree award",
    "Some college, no degree",
    "High school diploma or equivalent",
    "No formal educational credential",
]

# Bachelor's or above - the natural cut for "requires a 4-year college
# degree," which is the immediate ask.
BACHELORS_OR_HIGHER = {"Doctoral or professional degree", "Master's degree", "Bachelor's degree"}

# Associate's or above - relevant for the priority-population thread
# (2-year degree students) once that's picked back up. Not used for
# filtering yet, just tagged now so it's there when needed.
ASSOCIATES_OR_HIGHER = BACHELORS_OR_HIGHER | {"Associate's degree"}


def fetch_education_crosswalk() -> pd.DataFrame:
    if LOCAL_XLSX_PATH:
        print(f"Reading local file: {LOCAL_XLSX_PATH}")
        with open(LOCAL_XLSX_PATH, "rb") as f:
            content = f.read()
    else:
        try:
            resp = requests.get(EDUCATION_XLSX_URL, headers=REQUEST_HEADERS, timeout=30)
            resp.raise_for_status()
            content = resp.content
        except requests.exceptions.HTTPError as e:
            raise RuntimeError(
                f"Download failed ({e}). BLS's site sometimes blocks scripted requests "
                f"outright regardless of headers. Easiest fix: open "
                f"{EDUCATION_XLSX_URL} in your browser, save it to your data folder, "
                f"then set LOCAL_XLSX_PATH at the top of this script to that file's "
                f"path and re-run - no code changes needed beyond that one variable."
            ) from e

    # This is a MULTI-SHEET workbook (Index, Table 5.1, 5.2, 5.3, 5.4) -
    # not a single-table file. The first sheet is just an index/title page
    # listing what's on each tab, so we need to find the sheet that
    # actually contains "Typical education needed for entry", rather than
    # assuming sheet 0 is the data we want.
    all_sheets = pd.read_excel(BytesIO(content), sheet_name=None, header=None)
    print(f"Workbook sheets found: {list(all_sheets.keys())}")

    # Confirmed from the real file: the sheet we want is reliably named
    # "Table 5.4". Target it directly rather than searching by content -
    # Table 5.2 ALSO contains the phrase "Typical education needed for
    # entry" in its own header (it's grouped BY that category rather than
    # being a per-occupation crosswalk), which caused a false match when
    # searching sheet-by-sheet for that phrase alone.
    if "Table 5.4" in all_sheets:
        target_sheet_name = "Table 5.4"
        raw = all_sheets[target_sheet_name]
        header_row_idx = None
        for i, row in raw.iterrows():
            if row.astype(str).str.contains("National Employment Matrix code", case=False, na=False).any():
                header_row_idx = i
                break
        if header_row_idx is None:
            raise ValueError("Found 'Table 5.4' but couldn't locate its header row - inspect the sheet manually.")
    else:
        # Fallback: search every sheet, but require BOTH the education
        # column AND a code column on the same header row, so we don't
        # false-match on Table 5.2 again if the sheet naming ever changes.
        target_sheet_name, header_row_idx = None, None
        for sheet_name, raw in all_sheets.items():
            for i, row in raw.iterrows():
                row_text = row.astype(str)
                has_education = row_text.str.contains("Typical education needed for entry", case=False, na=False).any()
                has_code = row_text.str.contains("matrix code", case=False, na=False).any()
                if has_education and has_code:
                    target_sheet_name, header_row_idx = sheet_name, i
                    break
            if target_sheet_name:
                break
        if target_sheet_name is None:
            raise ValueError(
                "Couldn't find a sheet with both an education column and a code column - "
                "BLS may have changed the format. Inspect the raw file manually."
            )

    print(f"Found the data on sheet '{target_sheet_name}', header row {header_row_idx}")

    df = pd.read_excel(BytesIO(content), sheet_name=target_sheet_name, header=header_row_idx)
    df.columns = [str(c).strip() for c in df.columns]

    code_col = next((c for c in df.columns if "code" in c.lower()), None)
    edu_col = next((c for c in df.columns if "typical education" in c.lower()), None)
    if code_col is None or edu_col is None:
        raise ValueError(f"Couldn't identify code/education columns. Columns found: {df.columns.tolist()}")

    crosswalk = df[[code_col, edu_col]].rename(
        columns={code_col: "SOC Code", edu_col: "Typical Education Needed For Entry"}
    )
    crosswalk["SOC Code"] = crosswalk["SOC Code"].astype(str).str.strip()
    crosswalk = crosswalk.dropna(subset=["SOC Code", "Typical Education Needed For Entry"])
    return crosswalk


def main():
    print("Downloading BLS education crosswalk...")
    crosswalk = fetch_education_crosswalk()
    print(f"Crosswalk loaded: {len(crosswalk)} SOC codes")
    print(f"Education level distribution in the crosswalk itself:")
    print(crosswalk["Typical Education Needed For Entry"].value_counts().to_string())

    coded = pd.read_csv(CODED_PATH, encoding="utf-8-sig")
    coded["SOC Code"] = coded["SOC Code"].astype(str).str.strip()

    tagged = coded.merge(crosswalk, on="SOC Code", how="left")

    n_unmatched = tagged["Typical Education Needed For Entry"].isna().sum()
    n_total_coded = coded["SOC Code"].notna().sum()  # only count rows that had a real SOC code to begin with
    print(f"\nRows with a SOC code but no crosswalk match: {n_unmatched} "
          f"(these are likely the '00-9900 Insufficient Information' sentinel rows, "
          f"which correctly have nothing to match)")

    tagged["Requires Bachelors Or Higher"] = tagged["Typical Education Needed For Entry"].isin(BACHELORS_OR_HIGHER)
    tagged["Requires Associates Or Higher"] = tagged["Typical Education Needed For Entry"].isin(ASSOCIATES_OR_HIGHER)

    tagged.to_csv(OUT_PATH, index=False)
    print(f"\nWrote {OUT_PATH} ({len(tagged)} rows)")

    successfully_coded = tagged[tagged["API Call Error"].isna()]
    n_bachelors = successfully_coded["Requires Bachelors Or Higher"].sum()
    n_valid = successfully_coded["Typical Education Needed For Entry"].notna().sum()
    print(f"\nOf {len(successfully_coded)} successfully coded placements:")
    print(f"  {n_valid} matched an education requirement")
    print(f"  {n_bachelors} ({n_bachelors / n_valid * 100:.1f}% of matched) typically require a Bachelor's degree or higher")

    print("\nFull education-level breakdown, successfully coded placements:")
    print(successfully_coded["Typical Education Needed For Entry"].value_counts(dropna=False).to_string())

    return tagged


if __name__ == "__main__":
    main()

Reading local file: ../data/education.xlsx
Workbook sheets found: ['Index', 'Table 5.1', 'Table 5.2', 'Table 5.3', 'Table 5.4']
Found the data on sheet 'Table 5.4', header row 1
Crosswalk loaded: 832 SOC codes
Education level distribution in the crosswalk itself:
Typical Education Needed For Entry
High school diploma or equivalent    326
Bachelor's degree                    178
No formal educational credential     109
Doctoral or professional degree       73
Postsecondary nondegree award         51
Associate's degree                    48
Master's degree                       40
Some college, no degree                7

Rows with a SOC code but no crosswalk match: 1299 (these are likely the '00-9900 Insufficient Information' sentinel rows, which correctly have nothing to match)

Wrote ../data/AllRoles_nioccs_coded_with_education.csv (8582 rows)

Of 7965 successfully coded placements:
  7283 matched an education requirement
  4716 (64.8% of matched) typically require a Bachelor's degree